In [ ]:
from pydantic import BaseModel, AnyUrl, ConfigDict, Field, field_validator
from typing import List, Optional
from dotenv import load_dotenv
import os

load_dotenv()

class CensusQueryParams(BaseModel):
    model_config = ConfigDict(populate_by_name=True)

    # The Census Bureau uses 'get' for variable lists
    get_vars: List[str] = Field(..., alias="get")
    # 'for' is used for the geographic level (e.g., 'state:06')
    geo_for: str = Field(..., alias="for")
    # Optional 'in' for nesting geographies (e.g., 'county:001')
    geo_in: Optional[str] = Field(None, alias="in")
    # Your 40-character Census API key
    key: str = Field(..., min_length=40, max_length=40)

    @field_validator("get_vars", mode="before")
    @classmethod
    def split_vars(cls, v):
        if isinstance(v, str):
            return [var.strip() for var in v.split(",")]
        return v


In [ ]:
print(os.getenv("CENSUS_API_KEY")[:7])

94ade89


In [ ]:
# Census API requires a full dataset path. For ACS variables (e.g. B01001_001E) use the ACS 5-year endpoint.
DEFAULT_CENSUS_DATASET = "https://api.census.gov/data/2023/acs/acs5"


class DefaultCensusDataset(BaseModel):
    """Validates the Census API dataset URL (e.g. base path to a specific survey)."""
    root_url: AnyUrl = Field(..., description="Full dataset URL for the Census API (e.g. .../data/2023/acs/acs5)")


class CensusApiResponse(BaseModel):
    """Census API returns [[headers], [row1], [row2], ...]. This model parses and validates that shape."""

    headers: List[str] = Field(..., description="Column names from the first row of the API response")
    rows: List[List[str]] = Field(..., description="Data rows (list of cell values per row)")

    @classmethod
    def from_api_payload(cls, raw: list) -> "CensusApiResponse":
        if not raw or not isinstance(raw[0], list):
            raise ValueError("Census API response must be a non-empty list of lists")
        headers = [str(c) for c in raw[0]]
        rows = [list(map(str, row)) for row in raw[1:]]
        for i, row in enumerate(rows):
            if len(row) != len(headers):
                raise ValueError(f"Row {i} has {len(row)} values, expected {len(headers)}")
        return cls(headers=headers, rows=rows)

    def to_records(self) -> List[dict]:
        """Return rows as a list of dicts keyed by header (e.g. for DataFrame or iteration)."""
        return [dict(zip(self.headers, row)) for row in self.rows]

In [ ]:
import httpx


def get_census_data(
    params_dict: dict,
    dataset_url: str | DefaultCensusDataset = DEFAULT_CENSUS_DATASET,
) -> CensusApiResponse:
    # 1. Validate query params and dataset URL
    validated_params = CensusQueryParams(**params_dict)
    url_model = (
        dataset_url if isinstance(dataset_url, DefaultCensusDataset)
        else DefaultCensusDataset(root_url=dataset_url)
    )

    # 2. Request and parse response
    response = httpx.get(
        str(url_model.root_url),
        params=validated_params.model_dump(by_alias=True, exclude_none=True)
    )
    response.raise_for_status()
    raw = response.json()

    return CensusApiResponse.from_api_payload(raw)

# Example Usage
my_query = {
    "get": "NAME,B01001_001E",
    "for": "state:06",
    "key": os.getenv("CENSUS_API_KEY")
}




In [ ]:
data = get_census_data(my_query)

In [ ]:
data

[['NAME', 'state'], ['California', '06']]